# Hướng 1: Kiến trúc có sẵn — ResNet-18 pretrained (Transfer Learning)

Phân loại biển báo giao thông Việt Nam thành 5 nhóm (P, W, R, I, S), ảnh đầu vào **32x32x3**.
Làm theo các bước transfer learning của bài **05b / 07** trên lớp, nhưng thay VGG16 bằng **ResNet-18**: lấy ResNet-18 đã train trên ImageNet, **đóng băng** (freeze) phần trích đặc trưng, thêm lớp phân loại mới, train lớp mới trước rồi **mở băng** (unfreeze) để fine-tune với learning rate nhỏ.

Trước khi chạy, upload lên Google Drive thư mục `MyDrive/bienbao/`:
- `cnn_dataset_32.zip` (nén từ `Data_Processed/cnn_dataset_32`, tạo bằng `Data_Pipeline/build_cnn_crops.py`)

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từng cell.
Model và lịch sử train được lưu vào `MyDrive/bienbao/ket_qua/`.

In [1]:
!nvidia-smi

Sun Nov  1 17:31:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.107.02             Driver Version: 550.107.02     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   40C    P0              7W /   35W |       5MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

try:  # Chạy trên Colab: lấy dataset từ Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/bienbao'
    if not os.path.exists('/content/cnn_dataset_32'):
        !unzip -q {DRIVE_DIR}/cnn_dataset_32.zip -d /content/
    DATA_DIR = '/content/cnn_dataset_32/'
    OUT_DIR = DRIVE_DIR + '/ket_qua/'
except ImportError:  # Chạy trên máy
    DATA_DIR = '../../Data_Processed/cnn_dataset_32/'
    OUT_DIR = '../Inference/'
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
import glob
import json
import time

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

## Dữ liệu
Ảnh crop biển báo 32x32, chia sẵn train/val/test. Toàn bộ ảnh được đưa lên GPU một lần (giống bài 03).

In [4]:
IMG_HEIGHT = 32
IMG_WIDTH = 32
IMG_CHS = 3

# Tên thư mục = nhãn (lớp IE đã bỏ vì quá ít ảnh)
DATA_LABELS = ['0_P', '1_W', '2_R', '3_I', '5_S']
N_CLASSES = len(DATA_LABELS)

class MyDataset(Dataset):
    def __init__(self, data_dir):
        imgs = []
        labels = []
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = sorted(glob.glob(data_dir + label + '/*.png'))
            for path in data_paths:
                imgs.append(tv_io.read_image(path, tv_io.ImageReadMode.RGB))  # 3 x 32 x 32
                labels.append(l_idx)
        self.xs = (torch.stack(imgs).float() / 255).to(device)  # Normalize values from 0 to 1
        self.ys = torch.tensor(labels).to(device)

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]

    def __len__(self):
        return len(self.xs)

In [5]:
n = 64

train_data = MyDataset(DATA_DIR + 'train/')
train_loader = DataLoader(train_data, batch_size=n, shuffle=True)
train_N = len(train_loader.dataset)

valid_data = MyDataset(DATA_DIR + 'val/')
valid_loader = DataLoader(valid_data, batch_size=n)
valid_N = len(valid_loader.dataset)

test_data = MyDataset(DATA_DIR + 'test/')
test_loader = DataLoader(test_data, batch_size=n)
test_N = len(test_loader.dataset)

print('Train/Valid/Test:', train_N, valid_N, test_N)
train_data.xs.shape

Train/Valid/Test: 29317 3974 2259


torch.Size([29317, 3, 32, 32])

## Load ResNet-18 pretrained
ResNet-18 gồm lớp Conv 7x7 + MaxPool, 4 tầng residual block (64 → 128 → 256 → 512 kênh), `avgpool` và lớp `fc` 1000 lớp ImageNet.
Ảnh 32x32 đi qua mạng còn **512 x 1 x 1**, `avgpool` cho ra vector 512 đặc trưng. Ta bỏ lớp `fc` gốc (thay bằng `nn.Identity()`) rồi thêm lớp phân loại mới cho 5 nhóm biển.

ResNet-18 được train với ảnh đã chuẩn hoá theo mean/std của ImageNet, nên lớp đầu của model là `Normalize`.

In [6]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

# load the ResNet-18 network *pre-trained* on the ImageNet dataset
weights = ResNet18_Weights.DEFAULT
resnet_model = resnet18(weights=weights)
resnet_model.fc = nn.Identity()  # bỏ lớp phân loại 1000 lớp của ImageNet -> ra 512 đặc trưng

# Freeze base model
resnet_model.requires_grad_(False)
print("ResNet-18 Frozen")

ResNet-18 Frozen


In [7]:
# Input 3 x 32 x 32
my_model = nn.Sequential(
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # chuẩn hoá như ImageNet
    resnet_model,  # 512
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(.3),
    nn.Linear(256, N_CLASSES)
)
my_model = my_model.to(device)

total = sum(p.numel() for p in my_model.parameters())
trainable = sum(p.numel() for p in my_model.parameters() if p.requires_grad)
print(f'Tổng tham số: {total:,} | đang train: {trainable:,}')

Tổng tham số: 11,309,125 | đang train: 132,613


In [8]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(my_model.parameters())

## Data augmentation
Giống bài 04a nhưng **không lật ngang** (`RandomHorizontalFlip`), vì lật biển "cấm rẽ trái" sẽ thành "cấm rẽ phải".

In [9]:
random_trans = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop((IMG_WIDTH, IMG_HEIGHT), scale=(.85, 1), ratio=(1, 1)),
    transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.2),
])

## Hàm train / validate (giống bài 03)

In [10]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N


def train(model):
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(random_trans(x))
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    loss = loss / len(train_loader)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))
    return loss, accuracy


def validate(model, loader, N, name='Valid'):
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, N)
    loss = loss / len(loader)
    print('{} - Loss: {:.4f} Accuracy: {:.4f}'.format(name, loss, accuracy))
    return loss, accuracy


history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

def run_epochs(model, epochs):
    for epoch in range(epochs):
        print('Epoch: {}'.format(len(history['train_loss'])))
        train_loss, train_acc = train(model)
        valid_loss, valid_acc = validate(model, valid_loader, valid_N)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['valid_loss'].append(valid_loss)
        history['valid_acc'].append(valid_acc)

## Bước 1: Train lớp phân loại mới (ResNet-18 đang freeze)

In [11]:
start = time.time()
run_epochs(my_model, 5)

Epoch: 0


Train - Loss: 0.6510 Accuracy: 0.7747


Valid - Loss: 0.4628 Accuracy: 0.8530
Epoch: 1


Train - Loss: 0.5488 Accuracy: 0.8104


Valid - Loss: 0.4774 Accuracy: 0.8440
Epoch: 2


Train - Loss: 0.5177 Accuracy: 0.8203


Valid - Loss: 0.4311 Accuracy: 0.8639
Epoch: 3


Train - Loss: 0.4972 Accuracy: 0.8287


Valid - Loss: 0.4173 Accuracy: 0.8639
Epoch: 4


Train - Loss: 0.4846 Accuracy: 0.8289


Valid - Loss: 0.3810 Accuracy: 0.8767


## Bước 2: Unfreeze ResNet-18 và fine-tune với learning rate nhỏ

In [12]:
# Unfreeze the base model
resnet_model.requires_grad_(True)
optimizer = Adam(my_model.parameters(), lr=.0001)

In [13]:
run_epochs(my_model, 35)
train_time = time.time() - start
print(f'Thời gian train: {train_time / 60:.1f} phút')

Epoch: 5


Train - Loss: 0.2456 Accuracy: 0.9194


Valid - Loss: 0.1035 Accuracy: 0.9703
Epoch: 6


Train - Loss: 0.1400 Accuracy: 0.9522


Valid - Loss: 0.0854 Accuracy: 0.9746
Epoch: 7


Train - Loss: 0.1035 Accuracy: 0.9639


... (61 outputs left out)

Train - Loss: 0.0126 Accuracy: 0.9957


Valid - Loss: 0.0554 Accuracy: 0.9882
Epoch: 39


Train - Loss: 0.0226 Accuracy: 0.9966


Valid - Loss: 0.0692 Accuracy: 0.9836
Thời gian train: 11.7 phút


In [14]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['valid_acc'], label='Valid')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['valid_loss'], label='Valid')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
plt.show()

/tmp/ipykernel_10930/1506016483.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Đánh giá trên tập test và lưu model

In [15]:
test_loss, test_acc = validate(my_model, test_loader, test_N, name='Test')

history['train_time_s'] = train_time
history['test_acc'] = test_acc
torch.save(my_model, OUT_DIR + 'resnet18_pretrained.pth')
json.dump(history, open(OUT_DIR + 'resnet18_pretrained_history.json', 'w'))
print('Đã lưu', OUT_DIR + 'resnet18_pretrained.pth')

Test - Loss: 0.0185 Accuracy: 0.9947
Đã lưu ../Inference/resnet18_pretrained.pth
